# 01. 공통 전처리
두 팀원이 같은 입력·정답·분할·평가 마스크를 사용하기 위한 노트북입니다. 위에서 아래로 실행합니다.

- 매시간을 기준으로 향후 24시간을 직접 예측합니다. 정답은 **24시간 × 시간당 4개 전력값**입니다.
- 미래 생산계획·달력, 직전 한 시간 전력, 1·2주 전 같은 24시간의 전력·생산계획·공휴일 정보를 사용합니다.
- 날씨, 평균, 공장인원, 전기요금, 인건비는 모델 입력에서 제외합니다.
- 스케일링은 모델 학습 노트북에서 학습 데이터에만 fit합니다. 여기서는 원 단위로 저장합니다.
- 결과는 `results/preprocessing_24h`에 저장합니다. 기존 전처리 결과는 유지합니다.

**시각 규칙:** 기준시각 09시는 09시 행부터 다음 날 08시 행까지를 예측합니다. 각 행의 15·30·45·60분은 행 시각 이후의 관측값입니다. 직전 행의 60분 값은 기준시각에 즉시 확보된다고 가정합니다. 생산계획 24시간 전체도 이때 확정된 입력입니다.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Jupyter 없이 스크립트로 검증할 때 사용

ROOT = Path.cwd()
if not (ROOT / 'data/okm_augumented_2021.csv').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/okm_augumented_2021.csv'
OUT = ROOT / 'results/preprocessing_24h'
POWER_COLS = ['15분', '30분', '45분', '60분']
MINUTES = [15, 30, 45, 60]
HORIZON = 24
VALID_START = pd.Timestamp('2021-07-01')
TEST_START = pd.Timestamp('2021-08-01')
SUSPECT_DATES = pd.to_datetime(['2021-07-13', '2021-07-15'])
# 기존 전처리에서 사용한 대한민국 2021년 공휴일 달력을 유지합니다.
HOLIDAYS = pd.to_datetime([
    '2021-01-01', '2021-02-11', '2021-02-12', '2021-02-13',
    '2021-03-01', '2021-05-05', '2021-05-19', '2021-06-06',
    '2021-08-15', '2021-08-16', '2021-09-20', '2021-09-21',
    '2021-09-22', '2021-10-03', '2021-10-04', '2021-10-09',
    '2021-10-11', '2021-12-25'
])
raw = pd.read_csv(DATA_PATH)
print('원본:', raw.shape)

## 1. 시간 복구와 원본 검증
날짜마다 24개 행이 있고 정상 시간과 행 순서가 일치하는지 확인한 뒤, 잘못된 시간만 0~23으로 복구합니다. 날짜를 삭제한 상태에서 shift하지 않습니다. 모델에 쓰는 생산계획·전력에 결측이나 무한값이 있으면 중단하여 확인합니다.

In [ ]:
data = raw.copy()
data['date'] = pd.to_datetime(data['날짜'].astype(str), format='%Y%m%d')
data['original_hour'] = data['시간']
row_hour = data.groupby('date', sort=False).cumcount()
valid_hour = data['시간'].between(0, 23) & data['시간'].eq(data['시간'].round())
assert data.groupby('date').size().eq(24).all(), '날짜별 행 수 확인 필요'
assert data.loc[valid_hour, '시간'].eq(row_hour[valid_hour]).all(), '원본 행 순서 확인 필요'
data['time_repaired'] = ~valid_hour
data.loc[~valid_hour, '시간'] = row_hour[~valid_hour]
data['origin'] = data['date'] + pd.to_timedelta(data['시간'], unit='h')
data = data.sort_values('origin').reset_index(drop=True)
assert data['origin'].is_unique
assert data['origin'].diff().dropna().eq(pd.Timedelta(hours=1)).all()
assert np.isfinite(data[POWER_COLS + ['생산량']].to_numpy(dtype=float)).all()
assert data[POWER_COLS + ['생산량']].ge(0).all().all()
data['suspect_date'] = data['date'].isin(SUSPECT_DATES)
# 11시 행 전체를 포함: 행 시작시각 기준 [8/28 18시, 8/29 12시).
data['special_period'] = data['origin'].ge('2021-08-28 18:00') & data['origin'].lt('2021-08-29 12:00')
calendar = pd.DataFrame({
    'month': data['origin'].dt.month,
    'day': data['origin'].dt.day,
    'hour': data['origin'].dt.hour,
    'weekday': data['origin'].dt.dayofweek,
    'weekend': data['origin'].dt.dayofweek.ge(5).astype(int),
    'holiday': data['date'].isin(HOLIDAYS).astype(int),
})
display(data.loc[data['time_repaired'], ['date', 'original_hour', '시간']].head())
print('복구 행:', int(data['time_repaired'].sum()))
print('범위:', data['origin'].min(), '~', data['origin'].max())

## 2. 기준시각별 24시간 입력과 정답
`h00`은 기준시각의 시간 행, `h23`은 23시간 뒤 행입니다. `w1`, `w2`는 각각 해당 대상 행의 168·336시간 전입니다. 최근 전력 4개는 모든 미래 시간에 대해 동일한 기준시각에서 확보한 값입니다.

머신러닝용 `X`는 펼친 표이고, 딥러닝용 `sequence`는 같은 미래·주간 특성을 `(표본, 24, 특성)`으로 묶습니다. 최근 전력은 별도의 `(표본, 4)` 배열입니다. 정답의 펼침 순서는 시간 → 15·30·45·60분입니다.

In [ ]:
# 시간축의 연속성을 검증했으므로 정수 위치로 정확한 시간 블록을 조회합니다.
starts = np.arange(336, len(data) - HORIZON + 1)
target_pos = starts[:, None] + np.arange(HORIZON)
w1_pos, w2_pos = target_pos - 168, target_pos - 336
power = data[POWER_COLS].to_numpy(dtype=float)
production = data['생산량'].to_numpy(dtype=float)
holiday = calendar['holiday'].to_numpy()
plan = production[target_pos]
sequence_parts = [plan[..., None]]
sequence_names = ['planned_production']
for name in calendar.columns:
    sequence_parts.append(calendar[name].to_numpy()[target_pos][..., None])
    sequence_names.append(name)
for week, pos in [(1, w1_pos), (2, w2_pos)]:
    sequence_parts.extend([
        power[pos], production[pos][..., None], holiday[pos][..., None],
        (plan - production[pos])[..., None],
        ((plan > 0) != (production[pos] > 0))[..., None],
        (holiday[target_pos] != holiday[pos])[..., None],
    ])
    sequence_names.extend([
        *[f'w{week}_power_{m}' for m in MINUTES],
        f'w{week}_production', f'w{week}_holiday',
        f'w{week}_production_difference', f'w{week}_production_state_mismatch',
        f'w{week}_holiday_mismatch',
    ])
sequence = np.concatenate(sequence_parts, axis=2).astype(float)
recent = power[starts - 1]
targets = power[target_pos]
recent_names = [f'recent_previous_hour_{m}' for m in MINUTES]
feature_names = recent_names + [f'h{h:02d}_{name}' for h in range(HORIZON) for name in sequence_names]
target_names = [f'h{h:02d}_power_{m}' for h in range(HORIZON) for m in MINUTES]
sample_ids = pd.Index(starts, name='sample_id')
X = pd.DataFrame(np.concatenate([recent, sequence.reshape(len(starts), -1)], axis=1), index=sample_ids, columns=feature_names)
y = pd.DataFrame(targets.reshape(len(starts), -1), index=sample_ids, columns=target_names)
print('X:', X.shape, '| y:', y.shape, '| sequence:', sequence.shape)

## 3. 시간순 분할과 공통 평가 표시
학습·검증·테스트 각각에 정답 24시간이 완전히 들어오는 표본만 사용합니다. 경계를 가로지르는 표본은 별도로 표시합니다. 검증·테스트의 과거 입력은 이전 분할을 참조할 수 있습니다.

- 주 검증: 정답 구간이 7/13·7/15와 겹치지 않는 표본
- 보조 검증: 전체 표본, 그리고 정답·최근 전력·1·2주 전 입력 모두 의심 날짜를 참조하지 않는 표본
- 운영 시각은 미정이므로 모든 정각을 저장하며, 추후 `origin_hour`로 공통 선택합니다.
- 최종 1~7월 재학습 데이터는 **후보**로 저장합니다. 이상 날짜 포함 정책은 아직 확정하지 않았으므로 `primary_eligible` 또는 `history_clean_eligible` 등을 선택한 후 학습합니다.

특정 시간의 성능을 평가할 때는 아래 `target_masks`를 정답과 같은 순서로 사용합니다. 표본 전체 포함 여부와 시간별 포함 여부를 구분합니다.

In [ ]:
meta = pd.DataFrame(index=sample_ids)
meta['origin'] = data['origin'].iloc[starts].to_numpy()
meta['target_end'] = meta['origin'] + pd.Timedelta(hours=HORIZON)
meta['origin_hour'] = meta['origin'].dt.hour
suspect = data['suspect_date'].to_numpy()
special = data['special_period'].to_numpy()
meta['target_suspect'] = suspect[target_pos].any(axis=1)
meta['recent_suspect'] = suspect[starts - 1]
meta['w1_suspect'] = suspect[w1_pos].any(axis=1)
meta['w2_suspect'] = suspect[w2_pos].any(axis=1)
meta['primary_eligible'] = ~meta['target_suspect']
meta['history_clean_eligible'] = ~meta[['target_suspect', 'recent_suspect', 'w1_suspect', 'w2_suspect']].any(axis=1)
meta['special_period_overlap'] = special[target_pos].any(axis=1)
meta['split'] = 'boundary_excluded'
meta.loc[meta['target_end'].le(VALID_START), 'split'] = 'train'
meta.loc[meta['origin'].ge(VALID_START) & meta['target_end'].le(TEST_START), 'split'] = 'validation'
meta.loc[meta['origin'].ge(TEST_START), 'split'] = 'test'
meta['final_retrain_candidate'] = meta['target_end'].le(TEST_START)
split_masks = {name: meta['split'].eq(name).to_numpy() for name in ['train', 'validation', 'test']}
split_masks['final_retrain_candidates'] = meta['final_retrain_candidate'].to_numpy()

# 기준은 겹친 윈도우가 아니라 1~6월 원본의 각 관측값을 딱 한 번 사용합니다.
threshold_source = data.loc[data['origin'].lt(VALID_START), POWER_COLS].to_numpy().ravel()
peak_threshold = float(np.quantile(threshold_source, 0.95))
hour_masks = {
    'suspect_target': suspect[target_pos],
    'special_period': special[target_pos],
    'weekend_production': (calendar['weekend'].to_numpy()[target_pos] == 1) & (plan > 0),
    'holiday_production': (holiday[target_pos] == 1) & (plan > 0),
    'zero_production': plan == 0,
    'production_both_weeks_zero': (plan > 0) & (production[w1_pos] == 0) & (production[w2_pos] == 0),
}
target_masks = {name: np.repeat(values, 4, axis=1) for name, values in hour_masks.items()}
for name, lo, hi in [('horizon_0_3', 0, 3), ('horizon_3_12', 3, 12), ('horizon_12_24', 12, 24)]:
    hours = (np.arange(24) >= lo) & (np.arange(24) < hi)
    target_masks[name] = np.broadcast_to(np.repeat(hours, 4), y.shape).copy()
target_masks['actual_peak'] = y.to_numpy() >= peak_threshold
split_summary = meta.groupby('split').agg(samples=('origin', 'size'), first_origin=('origin', 'min'), last_origin=('origin', 'max'), primary_samples=('primary_eligible', 'sum'), history_clean_samples=('history_clean_eligible', 'sum'))
display(split_summary)
print('고정 피크 기준(q95):', peak_threshold)

## 4. 정렬·누출·분할 검증
과거 블록의 마지막 관측도 기준시각 이전인지 확인합니다. 첫 표본과 마지막 표본을 포함해 원본 시각으로 직접 조회하여, 배열 구성과 정답 순서를 검증합니다.

In [ ]:
assert X.index.equals(y.index) and y.index.equals(meta.index)
assert np.isfinite(X.to_numpy()).all() and np.isfinite(y.to_numpy()).all()
assert X.columns.is_unique and y.columns.is_unique
assert meta['origin'].iloc[0] == pd.Timestamp('2021-01-15')
assert meta.loc[meta['split'].eq('train'), 'origin'].max() == pd.Timestamp('2021-06-30')
for pos in [w1_pos, w2_pos]:
    latest_observed = data['origin'].to_numpy()[pos[:, -1]] + np.timedelta64(60, 'm')
    assert (latest_observed <= meta['origin'].to_numpy()).all()
assert (data['origin'].to_numpy()[starts - 1] + np.timedelta64(60, 'm') == meta['origin'].to_numpy()).all()
lookup = data.set_index('origin')
for i in [0, len(starts) // 2, len(starts) - 1]:
    origin = meta['origin'].iloc[i]
    times = pd.date_range(origin, periods=24, freq='h')
    np.testing.assert_array_equal(targets[i], lookup.loc[times, POWER_COLS].to_numpy())
    np.testing.assert_array_equal(recent[i], lookup.loc[origin - pd.Timedelta(hours=1), POWER_COLS].to_numpy(dtype=float))
    for week in [1, 2]:
        cols = [sequence_names.index(f'w{week}_power_{m}') for m in MINUTES]
        np.testing.assert_array_equal(sequence[i][:, cols], lookup.loc[times - pd.Timedelta(weeks=week), POWER_COLS].to_numpy())
for name, mask in split_masks.items():
    assert mask.any(), f'{name}: 빈 분할'
assert meta.loc[split_masks['train'], 'target_end'].le(VALID_START).all()
assert meta.loc[split_masks['validation'], 'origin'].ge(VALID_START).all()
assert meta.loc[split_masks['validation'], 'target_end'].le(TEST_START).all()
assert meta.loc[split_masks['test'], 'origin'].ge(TEST_START).all()
assert meta['target_end'].max() == data['origin'].max() + pd.Timedelta(hours=1)
assert len(threshold_source) == 181 * 24 * 4
np.testing.assert_array_equal(X.iloc[:, 4:].to_numpy().reshape(sequence.shape), sequence)
print('원본 대응·과거 정보 시각·정답 경계·배열 순서 검증 통과')

## 5. 팀 공통 파일 저장
`X_*.csv`, `y_*.csv`, `metadata_*.csv`는 같은 `sample_id` 순서입니다. `arrays_*.npz`에는 같은 입력·정답의 배열, 1·2주 기준선, 시간별 평가 마스크를 저장합니다. `config.json`에 열 순서·피크 기준·원본 해시를 기록합니다.

`hourly_review.csv`는 비용·공장인원 등을 포함한 검토용 자료이며 **모델 입력에 합치지 않습니다**. 최종 재학습 후보에는 6월/7월 경계를 넘는 표본도 포함됩니다. 처음 모델을 선정할 때 이 파일을 사용하면 안 됩니다.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
for name, mask in split_masks.items():
    for prefix, frame in [('X', X), ('y', y), ('metadata', meta)]:
        frame.loc[mask].to_csv(OUT / f'{prefix}_{name}.csv', encoding='utf-8-sig', index=True)
    np.savez_compressed(
        OUT / f'arrays_{name}.npz',
        sample_id=sample_ids.to_numpy()[mask], sequence=sequence[mask], recent=recent[mask],
        y=targets[mask], baseline_w1=power[w1_pos][mask], baseline_w2=power[w2_pos][mask],
        baseline_weekly_mean=((power[w1_pos] + power[w2_pos]) / 2)[mask],
        **{key: value[mask] for key, value in target_masks.items()},
    )
meta.to_csv(OUT / 'all_sample_metadata.csv', encoding='utf-8-sig')
split_summary.to_csv(OUT / 'split_summary.csv', encoding='utf-8-sig')
data.to_csv(OUT / 'hourly_review.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'feature': feature_names}).to_csv(OUT / 'feature_catalog.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([
    {'target': f'h{h:02d}_power_{m}', 'hour_offset': h, 'minute_in_hour': m, 'observation_offset_minutes': h * 60 + m}
    for h in range(24) for m in MINUTES
]).to_csv(OUT / 'target_catalog.csv', index=False, encoding='utf-8-sig')
config = {
    'source_file': str(DATA_PATH.relative_to(ROOT)),
    'source_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'horizon_hours': 24, 'stride_hours': 1, 'history_hours': [168, 336],
    'feature_columns': feature_names, 'sequence_features': sequence_names,
    'recent_features': recent_names, 'target_columns': target_names,
    'flatten_order': 'hour first, then feature / power minute (15,30,45,60)',
    'peak_threshold': peak_threshold, 'peak_comparison': '>=',
    'threshold_source': 'unique raw observations Jan 1 through June 30; fixed after retraining',
    'threshold_observations': len(threshold_source),
    'validation_start': str(VALID_START), 'test_start': str(TEST_START),
    'holiday_dates': [str(d.date()) for d in HOLIDAYS],
    'suspect_dates': [str(d.date()) for d in SUSPECT_DATES],
    'primary_evaluation_mask': 'primary_eligible',
    'strict_evaluation_mask': 'history_clean_eligible',
    'special_period_row_interval': '[2021-08-28 18:00, 2021-08-29 12:00)',
    'operating_hour': None, 'final_retrain_suspect_policy': 'not yet selected; candidates only',
    'availability': '24h production plan known at origin; previous hour 60-minute power immediately available',
    'scaling': None, 'missing_value_policy': 'stop if required production or power is missing',
    'weather': 'excluded', 'random_seed_for_downstream_models': 42,
}
(OUT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
# 저장 후 다시 읽어 팀원이 불러올 때도 행 순서와 값이 유지되는지 확인합니다.
for name, mask in split_masks.items():
    saved_x = pd.read_csv(OUT / f'X_{name}.csv', index_col='sample_id')
    saved_y = pd.read_csv(OUT / f'y_{name}.csv', index_col='sample_id')
    assert saved_x.index.equals(saved_y.index)
    np.testing.assert_allclose(saved_x.to_numpy(), X.loc[mask].to_numpy())
    with np.load(OUT / f'arrays_{name}.npz') as saved:
        np.testing.assert_array_equal(saved['sample_id'], saved_x.index.to_numpy())
        np.testing.assert_allclose(saved['y'].reshape(len(saved_y), -1), saved_y.to_numpy())
print('저장 및 재로딩 검증 완료:', OUT)

## 6. 각자 모델 노트북에서 불러오기
머신러닝은 CSV의 `X`, `y`를 사용합니다. 딥러닝은 NPZ의 `sequence`, `recent`, `y`를 사용합니다. 입력 정보는 같습니다. 최근 전력은 별도 입력으로 연결하거나 24시간 축에 반복해서 결합할 수 있습니다.

검증 예측을 만든 다음 `primary_eligible`을 이용해 평가 표본을 고릅니다. 운영 시각이 정해지면 같은 마스크에 `origin_hour` 조건을 추가합니다. q95는 평가에만 사용하며 정답은 실제 전력 수치입니다. 사용자 허용 피크는 이후 예측값과 비교합니다.

```python
from pathlib import Path
import json
import pandas as pd
import numpy as np

path = Path('results/preprocessing_24h')
X_train = pd.read_csv(path / 'X_train.csv', index_col='sample_id')
y_train = pd.read_csv(path / 'y_train.csv', index_col='sample_id')
X_valid = pd.read_csv(path / 'X_validation.csv', index_col='sample_id')
y_valid = pd.read_csv(path / 'y_validation.csv', index_col='sample_id')
meta_valid = pd.read_csv(path / 'metadata_validation.csv', index_col='sample_id')
primary = meta_valid['primary_eligible'].astype(bool)
config = json.loads((path / 'config.json').read_text(encoding='utf-8'))
threshold = config['peak_threshold']
with np.load(path / 'arrays_train.npz') as arrays:
    sequence_train = arrays['sequence']
    recent_train = arrays['recent']
    target_train = arrays['y']
```

모델 선정에는 학습·7월 검증만 사용합니다. 최종 설정과 재학습의 이상 날짜 처리 정책을 확정한 후 `final_retrain_candidates`에서 표본을 선택하고, 8~9월 테스트를 평가합니다. 과거 전체 기간을 EDA에서 확인한 이력은 결과 해석에 함께 기록합니다.